### TODO:
- GRB006 will fail because the events are being handled through the obx logic in the script. I need to add something to fix that.
- Calculate % of units that are:
    - stimulus selective
    - rate selective
    - double peak'd
- Give Marsa some new data to fit the GLM on. For that, I need to give her:
    - neural data dataframe (len of df is n_units)
        - unit ID
        - spike times
    - behavior data dataframe (len of df is n_trials)
        - trial start
        - center poke
        - stim onsets
        - left poke
        - right poke

In [1]:
import numpy as np
import pandas as pd
from labdata.schema import SpikeSorting, UnitCount, Dataset, DatasetEvents
from chipmunk import Chipmunk  # type: ignore

/Users/gabriel/lib/ephys/.venv/lib/python3.11/site-packages/datajoint/plugin.py:4: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources  # requires setuptools<82
[2026-03-04 14:53:05,032][INFO]: DataJoint 0.14.9 connected to rojasbowe@churchland-data.cmojfwfr0b9t.us-west-2.rds.amazonaws.com:3306


In [2]:
sess = (Chipmunk() & "setting_modalities = 'visual'") & SpikeSorting().fetch(
    "subject_name", "session_name", as_dict=True
)

neural_sess = (
    (Dataset() & "dataset_name LIKE 'ephys%'")
    & SpikeSorting().fetch("subject_name", "session_name", as_dict=True)
    & sess.fetch("subject_name", "session_name", as_dict=True)
)

behavior_data = pd.DataFrame(
    (
        sess.proj()
        * Chipmunk.Trial().proj("response", "rewarded")
        * Chipmunk.TrialParameters().proj("stim_rate_vision")
    ).fetch(as_dict=True)
)

neural_data = pd.DataFrame(
    (
        (
            SpikeSorting().Unit()
            & neural_sess.fetch("subject_name", "session_name", as_dict=True)
        )
        * (UnitCount.Unit() & "unit_criteria_id = 1" & "passes = 1")
    ).fetch(
        "subject_name",
        "session_name",
        "dataset_name",
        "unit_id",
        "spike_times",
        as_dict=True,
    )
)

In [3]:
z = []
for key in neural_sess:
    events = DatasetEvents.Digital() & key
    if len(events) > 1:
        events = pd.DataFrame(events.fetch_synced())
        align_ev = {
            "stim": events.query("event_name == '0'").event_timestamps.values[0],
            "trial_start": events.query("event_name == '2'").event_timestamps.values[0],
            "left_port": events.query("event_name == '4'").event_timestamps.values[0],
            "center_port": events.query("event_name == '5'").event_timestamps.values[0],
            "right_port": events.query(
                "event_name == '6' & stream_name == 'obx'"
            ).event_timestamps.values[0],
        }

        stim = align_ev["stim"]
        stim_ev = np.concatenate([[stim[0]], stim[1:][np.diff(stim) > 0.025]])
        first_stim_ev = np.concatenate([[stim[0]], stim[1:][np.diff(stim) > 1]])
        align_ev.update({"stim_ev": stim_ev, "first_stim_ev": first_stim_ev})
        z.append(
            {
                "subject_name": key["subject_name"],
                "session_name": key["session_name"],
                **align_ev,
            }
        )

obx_data = pd.DataFrame(z)

In [5]:
from pathlib import Path

out_dir = Path("/Users/gabriel/Desktop/data_export")
out_dir.mkdir(parents=True, exist_ok=True)

exclude_subjects = {"GRB006"}

dfs = {
    "behavior_data": behavior_data,
    "neural_data": neural_data,
    "obx_data": obx_data,
}

for name, df in dfs.items():
    export_df = df.copy()
    if "subject_name" in export_df.columns:
        export_df = export_df[~export_df["subject_name"].isin(exclude_subjects)]

    if {"subject_name", "session_name"}.issubset(export_df.columns):
        for (subject, session), sub_df in export_df.groupby(
            ["subject_name", "session_name"], dropna=False
        ):
            file_path = out_dir / f"{name}_{subject}_{session}.csv"
            sub_df.to_csv(file_path, index=False)
    else:
        file_path = out_dir / f"{name}.csv"
        export_df.to_csv(file_path, index=False)

print(f"Saved CSV files to: {out_dir} (excluded: {', '.join(exclude_subjects)})")

Saved CSV files to: /Users/gabriel/Desktop/data_export (excluded: GRB006)
